# Spike — validação da API do GitHub (Agente de Automação)

Testa autenticação via Personal Access Token e uma chamada simples de leitura (confirmar identidade do usuário), antes de qualquer operação de escrita (branch, commit, PR).

Referências: fluxo de Nível A (branch → commit → push → PR aberto, merge manual) já decidido.

In [0]:
import requests

token = dbutils.secrets.get(scope="pulse-secrets", key="github-pat")

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
}

resposta = requests.get("https://api.github.com/user", headers=headers)

print("Status code:", resposta.status_code)
print("Usuário autenticado:", resposta.json().get("login"))

In [0]:
from src.observabilidade.agente_automacao_git import AgenteAutomacaoGit

agente = AgenteAutomacaoGit(
    dbutils=dbutils,
    owner="brunofqueles",
    repo="poc-pulse-health-observability",
)

resultado = agente.criar_branch("teste/agente-automacao-spike")
print(resultado)

In [0]:
resultado_commit = agente.commitar_arquivo(
    branch="teste/agente-automacao-spike",
    caminho_arquivo="docs/teste_agente_automacao.md",
    conteudo="# Teste do Agente de Automação\n\nEste arquivo foi criado automaticamente via API do GitHub, pelo AgenteAutomacaoGit — spike de validação, Nível A (branch/commit/PR, merge manual).",
    mensagem="test: valida commit automatico via AgenteAutomacaoGit",
)
print(resultado_commit)

In [0]:
resultado_pr = agente.abrir_pr(
    branch="teste/agente-automacao-spike",
    titulo="test: valida AgenteAutomacaoGit (spike)",
    descricao="PR de teste, aberto automaticamente pelo AgenteAutomacaoGit. Valida os 3 métodos do Nível A: criar_branch, commitar_arquivo, abrir_pr. Merge continua manual, por decisão de design.",
)
print(resultado_pr)

In [0]:
with open(
    "/Workspace/Users/bruno.quelestech@outlook.com/poc-pulse-health-observability/docs/adr/adr-018-agente-automacao-git.md",
    "r", encoding="utf-8"
) as f:
    conteudo_adr = f.read()

resultado = agente.publicar_mudanca(
    nome_branch="docs/adr-018-agente-automacao-git",
    caminho_arquivo="docs/adr/adr-018-agente-automacao-git.md",
    conteudo=conteudo_adr,
    mensagem_commit="docs: adiciona ADR-018 (Agente de Automacao Git)",
    titulo_pr="docs: ADR-018 - Agente de Automacao Git",
    descricao_pr="Documenta o desenho, decisoes (Nivel A, MVP 1 arquivo por commit) e validacao do AgenteAutomacaoGit. Publicado via publicar_mudanca — primeiro uso real do agente, nao mais teste isolado.",
)
print(resultado)